In [4]:
import time,random
from pathlib import Path
import pandas as pd
import requests

BASE_URL="https://iss.moex.com/iss"
TICKERS= ['AFKS', 'AFLT', 'AGRO', 'AKRN', 'ALRS', 'ASTR', 'BANE', 'BANEP', 'BSPB', 'CBOM', 'CHMF', 'CNRU', 'DIXY', 'DOMRF', 'DSKY', 'ENPG', 'EONR', 'EPLN', 'FEES', 'FIVE', 'FIXP', 'FLOT', 'GAZP', 'GCHE', 'GLTR', 'GMKN', 'HEAD', 'HHRU', 'HYDR', 'IRAO', 'KMAZ', 'LEAS', 'LENT', 'LKOH', 'LNTA', 'LSRG', 'MAGN', 'MAIL', 'MDMG', 'MFON', 'MGNT', 'MOEX', 'MRKH', 'MSNG', 'MSRS', 'MSTT', 'MTLR', 'MTLRP', 'MTSS', 'MVID', 'NKNC', 'NLMK', 'NMTP', 'NVTK', 'OGKB', 'OZON', 'PGIL', 'PHOR', 'PHST', 'PIKK', 'PLZL', 'POGR', 'POLY', 'POSI', 'QIWI', 'RAGR', 'RASP', 'RENI', 'RNFT', 'ROSN', 'RSTI', 'RTKM', 'RTKMP', 'RUAL', 'RUALR', 'SBER', 'SBERP', 'SELG', 'SFIN', 'SGZH', 'SMLT', 'SNGS', 'SNGSP', 'SVAV', 'SVCB', 'T', 'TATN', 'TATNP', 'TCSG', 'TRMK', 'TRNFP', 'UGLD', 'UPRO', 'URKA', 'UWGN', 'VKCO', 'VSMO', 'VTBR', 'VZRZ', 'X5', 'YDEX', 'YNDX']
INTERVALS={"1m":1,"10m":10,"1h":60,"1d":24,"1w":7,"1mo":31,"1q":4}
FREQUENCY="1h"
RAW_DATA_DIR=Path("Raw_Data")

class MoexParser:
    def __init__(self,timeout=30,retries=5,pause=0.05):
        self.timeout=timeout
        self.retries=retries
        self.pause=pause
        self.session=requests.Session()
        self.session.headers.update({"User-Agent":"MOEX-OHLCV-Parser","Accept":"application/json"})

    def _request(self,url,params):
        error=None
        for attempt in range(self.retries):
            try:
                r=self.session.get(url,params=params,timeout=self.timeout)
                if r.status_code==200:return r.json()
                if r.status_code in {429,500,502,503,504}:
                    time.sleep(2**attempt+random.uniform(0,0.5))
                    continue
                r.raise_for_status()
            except requests.RequestException as e:
                error=e
                if attempt<self.retries-1:time.sleep(2**attempt+random.uniform(0,0.5))
        raise RuntimeError(error)

    def get_candles(self,ticker,interval=24):
        url=f"{BASE_URL}/engines/stock/markets/shares/securities/{ticker}/candles.json"
        start=0
        rows_all=[]
        columns=None

        while True:
            params={"iss.meta":"off","iss.only":"candles","interval":interval,"start":start}
            data=self._request(url,params)
            candles=data.get("candles")

            if candles is None:raise RuntimeError(f"{ticker}: candles отсутствует")

            columns=candles["columns"]
            rows=candles["data"]

            if not rows:break

            rows_all.extend(rows)
            start+=len(rows)
            print(f"{ticker}: {len(rows_all)}")
            time.sleep(self.pause)

        if not rows_all:return pd.DataFrame()

        df=pd.DataFrame(rows_all,columns=columns)
        df.insert(0,"ticker",ticker)

        for c in ["begin","end"]:
            if c in df:df[c]=pd.to_datetime(df[c],errors="coerce")

        for c in ["open","high","low","close","volume","value"]:
            if c in df:df[c]=pd.to_numeric(df[c],errors="coerce")

        cols=["ticker","begin","open","high","low","close","volume","value","end"]
        df=df[[c for c in cols if c in df]]
        df=df.drop_duplicates(["ticker","begin"],keep="last")
        return df.sort_values("begin").reset_index(drop=True)

def save_parquet(df,ticker,frequency):
    RAW_DATA_DIR.mkdir(parents=True,exist_ok=True)
    path=RAW_DATA_DIR/f"{ticker}_{frequency}.parquet"
    df.to_parquet(path,index=False,engine="pyarrow",compression="snappy")
    return path

def download_all():
    parser=MoexParser()
    interval=INTERVALS[FREQUENCY]
    RAW_DATA_DIR.mkdir(parents=True,exist_ok=True)

    for i,ticker in enumerate(TICKERS,1):
        print(f"[{i}/{len(TICKERS)}] {ticker}")
        try:
            df=parser.get_candles(ticker,interval)

            if df.empty:
                print(f"{ticker}: no data")
                continue

            path=save_parquet(df,ticker,FREQUENCY)
            print(f"{ticker}: {len(df)} rows | {df['begin'].min()} -> {df['begin'].max()} | {path}")

        except Exception as e:
            print(f"{ticker}: ERROR {e}")

if __name__=="__main__":
    download_all()

[1/102] AFKS
AFKS: 500
AFKS: 1000
AFKS: 1500
AFKS: 2000
AFKS: 2500
AFKS: 3000
AFKS: 3500
AFKS: 4000
AFKS: 4500
AFKS: 5000
AFKS: 5500
AFKS: 6000
AFKS: 6500
AFKS: 7000
AFKS: 7500
AFKS: 8000
AFKS: 8500
AFKS: 9000
AFKS: 9500
AFKS: 10000
AFKS: 10500
AFKS: 11000
AFKS: 11500
AFKS: 12000
AFKS: 12500
AFKS: 13000
AFKS: 13500
AFKS: 14000
AFKS: 14500
AFKS: 15000
AFKS: 15500
AFKS: 16000
AFKS: 16500
AFKS: 17000
AFKS: 17500
AFKS: 18000
AFKS: 18500
AFKS: 19000
AFKS: 19500
AFKS: 20000
AFKS: 20500
AFKS: 21000
AFKS: 21500
AFKS: 22000
AFKS: 22500
AFKS: 23000
AFKS: 23500
AFKS: 24000
AFKS: 24500
AFKS: 25000
AFKS: 25500
AFKS: 26000
AFKS: 26500
AFKS: 27000
AFKS: 27500
AFKS: 28000
AFKS: 28500
AFKS: 29000
AFKS: 29500
AFKS: 30000
AFKS: 30500
AFKS: 31000
AFKS: 31500
AFKS: 32000
AFKS: 32500
AFKS: 33000
AFKS: 33500
AFKS: 34000
AFKS: 34500
AFKS: 35000
AFKS: 35500
AFKS: 36000
AFKS: 36500
AFKS: 37000
AFKS: 37500
AFKS: 38000
AFKS: 38500
AFKS: 39000
AFKS: 39500
AFKS: 40000
AFKS: 40500
AFKS: 41000
AFKS: 41500
AFKS: 42000

In [5]:
def download_imoex():
    parser = MoexParser()
    interval = INTERVALS[FREQUENCY]

    ticker = "IMOEX"
    url = f"{BASE_URL}/engines/stock/markets/index/securities/{ticker}/candles.json"

    start = 0
    rows_all = []

    while True:
        params = {
            "iss.meta": "off",
            "iss.only": "candles",
            "interval": interval,
            "start": start,
        }

        data = parser._request(url, params)
        candles = data.get("candles")

        if candles is None:
            raise RuntimeError("IMOEX: candles отсутствует")

        columns = candles["columns"]
        rows = candles["data"]

        if not rows:
            break

        rows_all.extend(rows)
        start += len(rows)

        print(f"IMOEX: {len(rows_all)}")
        time.sleep(parser.pause)

    if not rows_all:
        print("IMOEX: no data")
        return

    df = pd.DataFrame(rows_all, columns=columns)

    for c in ["begin", "end"]:
        if c in df:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    for c in ["open", "high", "low", "close", "volume", "value"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    cols = ["begin", "open", "high", "low", "close", "volume", "value", "end"]
    df = df[[c for c in cols if c in df]]

    df = (
        df
        .drop_duplicates(["begin"], keep="last")
        .sort_values("begin")
        .reset_index(drop=True)
    )

    path = save_parquet(df, ticker, FREQUENCY)

    print(
        f"IMOEX: {len(df)} rows | "
        f"{df['begin'].min()} -> {df['begin'].max()} | "
        f"{path}"
    )

    return df

In [6]:
imoex = download_imoex()

IMOEX: 500
IMOEX: 1000
IMOEX: 1500
IMOEX: 2000
IMOEX: 2500
IMOEX: 3000
IMOEX: 3500
IMOEX: 4000
IMOEX: 4500
IMOEX: 5000
IMOEX: 5500
IMOEX: 6000
IMOEX: 6500
IMOEX: 7000
IMOEX: 7500
IMOEX: 8000
IMOEX: 8500
IMOEX: 9000
IMOEX: 9500
IMOEX: 10000
IMOEX: 10500
IMOEX: 11000
IMOEX: 11500
IMOEX: 12000
IMOEX: 12500
IMOEX: 13000
IMOEX: 13500
IMOEX: 14000
IMOEX: 14500
IMOEX: 15000
IMOEX: 15500
IMOEX: 16000
IMOEX: 16500
IMOEX: 17000
IMOEX: 17500
IMOEX: 18000
IMOEX: 18500
IMOEX: 19000
IMOEX: 19500
IMOEX: 20000
IMOEX: 20500
IMOEX: 21000
IMOEX: 21500
IMOEX: 22000
IMOEX: 22500
IMOEX: 23000
IMOEX: 23500
IMOEX: 24000
IMOEX: 24500
IMOEX: 25000
IMOEX: 25500
IMOEX: 26000
IMOEX: 26500
IMOEX: 27000
IMOEX: 27500
IMOEX: 28000
IMOEX: 28500
IMOEX: 29000
IMOEX: 29500
IMOEX: 30000
IMOEX: 30500
IMOEX: 31000
IMOEX: 31500
IMOEX: 32000
IMOEX: 32500
IMOEX: 33000
IMOEX: 33500
IMOEX: 34000
IMOEX: 34500
IMOEX: 35000
IMOEX: 35500
IMOEX: 36000
IMOEX: 36261
IMOEX: 36261 rows | 2011-11-17 10:00:00 -> 2026-08-28 19:00:00 | Raw_Da

In [8]:
f = pd.read_parquet('moex_weights_panel.parquet')
f.shape

(60, 102)